# DataSage 1/4 — Cleaning Inference

Loads the **cleaning** GRPO LoRA adapter and runs inference against the live HF Space environment.  
Also runs the **base model** (no LoRA) on the same seeds for comparison.

| Setting | Value |
|---------|-------|
| Runtime | **GPU** (T4 minimum, A100 recommended) |
| Secrets | `HF_TOKEN` (optional, models are public) |
| LoRA adapter | `ricalanis/cleaning-grpo` |
| Base model | `unsloth/Qwen2.5-3B-Instruct` |
| Environment | `https://ricalanis-datasage-cleaning.hf.space` |
| Output | `cleaning_results.json` |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/01_cleaning_inference.ipynb)

In [ ]:
# Cell 1: Install dependencies
# Unsloth handles torch/CUDA setup automatically in Colab
!pip install -q unsloth trl requests numpy datasets huggingface_hub

In [ ]:
# Cell 2: Imports, Colab detection, GPU check
import os
import sys
import json
import re
import time
import requests
import numpy as np
import torch
from datetime import datetime

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("Loaded HF_TOKEN from Colab Secrets")
    except Exception:
        print("HF_TOKEN not in Secrets (OK — models are public)")
else:
    print("Running locally")

# GPU check
assert torch.cuda.is_available(), (
    "No GPU detected! Go to: Runtime > Change runtime type > GPU (T4 or better)"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: All configuration — edit these values if needed

# Model
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"  # base model (Unsloth 4-bit)
LORA_REPO  = "ricalanis/cleaning-grpo"        # GRPO fine-tuned LoRA adapter
MAX_SEQ_LENGTH = 2048                          # must match training config

# Environment
ENV_URL = "https://ricalanis-datasage-cleaning.hf.space"

# Inference settings
DOMAINS = ["hr", "sales", "pm", "it_ops"]   # 4 enterprise domains
EPISODES_PER_DOMAIN = 3                        # episodes per domain for LoRA model
BASE_EPISODES_PER_DOMAIN = 1                   # fewer for base model (slower)
MAX_STEPS = 15                                 # max cleaning steps per episode

# System prompt (same as used during GRPO training)
SYSTEM_PROMPT = (
    "You are a data quality agent. You clean enterprise datasets across multiple "
    "domains (HR, Sales, Project Management, IT Operations).\n\n"
    "Each turn, analyze the data and respond with a JSON cleaning action:\n"
    '{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}\n\n'
    "Available operations:\n"
    "- fill_null: Fill null values (value can be 'median', 'mode', or a specific value)\n"
    "- fix_type: Fix type mismatches in a column (cast to proper type)\n"
    "- remove_duplicate: Remove duplicate rows\n"
    "- standardize: Standardize column values (strip whitespace, normalize case)\n"
    "- trim: Trim whitespace from column values\n"
    "- correct_typo: Correct typos in categorical values\n\n"
    "Think step by step: examine the data quality report, identify the most "
    "impactful issue, then act."
)

print("Config loaded.")
print(f"  LoRA:       {LORA_REPO}")
print(f"  Base:       {BASE_MODEL}")
print(f"  Env:        {ENV_URL}")
print(f"  Domains:    {DOMAINS}")
print(f"  Episodes:   {EPISODES_PER_DOMAIN} per domain (LoRA), {BASE_EPISODES_PER_DOMAIN} (base)")
print(f"  Max steps:  {MAX_STEPS}")

In [ ]:
# Cell 4: Helper functions (action parser, env client, inference)
# All self-contained — no external imports.

import json
import re
import time
import requests
import numpy as np
import torch


# --- Action parser (inlined from training/shared/parsers.py) ---

def _extract_column(text):
    """Extract a column name from model output text."""
    quoted = re.findall(r'["\']([\w]+)["\']', text)
    if quoted:
        return quoted[0]
    camel = re.findall(r'\b([A-Z][a-z]+(?:[A-Z][a-z]+)+)\b', text)
    if camel:
        return camel[0]
    return ""


def parse_action(text):
    """Parse model text output into a cleaning action dict."""
    # Try JSON extraction first
    m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            if "operation" in data and "column" in data:
                return data
        except json.JSONDecodeError:
            pass
    # Keyword fallback
    tl = text.lower()
    if "fill" in tl or "null" in tl:
        return {"operation": "fill_null", "column": _extract_column(text), "value": "median", "params": {}}
    if "type" in tl or "cast" in tl:
        return {"operation": "fix_type", "column": _extract_column(text), "value": "numeric", "params": {}}
    if "duplicate" in tl:
        return {"operation": "remove_duplicate", "column": "", "params": {}}
    if "standard" in tl:
        return {"operation": "standardize", "column": _extract_column(text), "params": {}}
    if "trim" in tl:
        return {"operation": "trim", "column": _extract_column(text), "params": {}}
    if "typo" in tl:
        return {"operation": "correct_typo", "column": _extract_column(text), "params": {}}
    return {"operation": "fill_null", "column": "", "value": "median", "params": {}}


# --- Environment client (HTTP calls to HF Space) ---

def env_reset(seed=42):
    """Reset the cleaning environment. Returns observation dict."""
    try:
        r = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed}, timeout=30)
        r.raise_for_status()
        d = r.json()
        return d.get("observation", d)
    except Exception as e:
        print(f"  [WARN] env_reset failed: {e}")
        return {"error": str(e)}


def env_step(action):
    """Send an action to the environment. Returns {observation, reward, done, info}."""
    try:
        r = requests.post(f"{ENV_URL}/web/step", json={"action": action}, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"  [WARN] env_step failed: {e}")
        return {"error": str(e), "reward": 0, "done": True, "observation": {}}


def format_obs(obs):
    """Format an observation dict into a user prompt string."""
    return (
        f"Domain: {obs.get('domain', 'unknown')}\n"
        f"DQ Score: {obs.get('dq_score', 'N/A')}\n"
        f"DQ Report: {obs.get('dq_report', '')}\n"
        f"Columns: {obs.get('columns_info', '')}\n"
        f"Data Preview: {str(obs.get('data_preview', ''))[:500]}\n"
        f"Step {obs.get('step_number', '?')}/{obs.get('max_steps', '?')}\n\n"
        "Output a single JSON cleaning action."
    )


# --- Model inference ---

def generate(model, tokenizer, sys_prompt, user_prompt, max_new_tokens=256):
    """Generate a response using the chat template."""
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    # Strip thinking tags (Qwen sometimes adds these)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    return response


def run_episode(model, tokenizer, seed=42):
    """Run one full cleaning episode. Returns metrics dict."""
    obs = env_reset(seed)
    if "error" in obs:
        return {"error": obs["error"]}

    rewards = []
    actions = []
    initial_dq = obs.get("dq_score", 0)

    for step in range(MAX_STEPS):
        resp = generate(model, tokenizer, SYSTEM_PROMPT, format_obs(obs))
        action = parse_action(resp)
        actions.append(action)

        result = env_step(action)
        rewards.append(result.get("reward", 0))

        new_obs = result.get("observation", {})
        if isinstance(new_obs, dict):
            obs = {**obs, **new_obs}

        if result.get("done", False):
            break

    final_dq = obs.get("dq_score", initial_dq)
    return {
        "initial_dq": float(initial_dq),
        "final_dq": float(final_dq),
        "dq_improvement": float(final_dq - initial_dq),
        "rewards": [float(r) for r in rewards],
        "avg_reward": float(np.mean(rewards)) if rewards else 0.0,
        "steps": len(rewards),
        "actions": actions,
    }


# --- Verify environment is reachable ---
try:
    r = requests.get(ENV_URL, timeout=10)
    print(f"Environment OK: {ENV_URL} (status {r.status_code})")
except Exception as e:
    print(f"WARNING: Environment UNREACHABLE: {ENV_URL} ({e})")

print("All helpers defined.")

In [ ]:
# Cell 5: Load LoRA model
from unsloth import FastLanguageModel

print(f"Loading LoRA adapter: {LORA_REPO}")
print(f"  (base: {BASE_MODEL}, 4-bit quantized)")
t0 = time.time()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_REPO,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.6,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_inference(model)

print(f"Loaded in {time.time()-t0:.1f}s")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 6: Run LoRA inference — 4 domains x 3 episodes = 12 episodes
print("=" * 60)
print(f"CLEANING — DataSage LoRA ({LORA_REPO})")
print(f"  {len(DOMAINS)} domains x {EPISODES_PER_DOMAIN} episodes = {len(DOMAINS)*EPISODES_PER_DOMAIN} total")
print("=" * 60)

lora_results = []
for di, domain in enumerate(DOMAINS):
    for ep in range(EPISODES_PER_DOMAIN):
        seed = di * 100 + ep + 1
        print(f"  [{domain}] ep {ep+1}/{EPISODES_PER_DOMAIN} seed={seed}", end=" ... ")
        t0 = time.time()

        result = run_episode(model, tokenizer, seed)
        result["domain"] = domain
        result["seed"] = seed
        result["latency_s"] = round(time.time() - t0, 1)
        lora_results.append(result)

        if "error" in result:
            print(f"ERROR: {result['error']}")
        else:
            print(f"DQ={result['final_dq']:.4f} R={result['avg_reward']:.4f} steps={result['steps']} ({result['latency_s']}s)")

# Summary
valid = [r for r in lora_results if "error" not in r]
if valid:
    print(f"\n--- LoRA Summary ({len(valid)} episodes) ---")
    print(f"  Mean DQ:     {np.mean([r['final_dq'] for r in valid]):.4f}")
    print(f"  Mean Reward: {np.mean([r['avg_reward'] for r in valid]):.4f}")
    print(f"  Mean Steps:  {np.mean([r['steps'] for r in valid]):.1f}")

# Free GPU memory before loading base model
del model, tokenizer
torch.cuda.empty_cache()
print(f"\nGPU freed. VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 7: Load base model (no LoRA) for comparison
from unsloth import FastLanguageModel

print(f"Loading base model: {BASE_MODEL}")
t0 = time.time()

base_m, base_t = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.6,
)
if base_t.pad_token is None:
    base_t.pad_token = base_t.eos_token

FastLanguageModel.for_inference(base_m)
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Cell 8: Run base model inference — 1 episode per domain
print(f"\nBASE MODEL — Cleaning ({BASE_MODEL})")
print(f"  {len(DOMAINS)} domains x {BASE_EPISODES_PER_DOMAIN} episode = {len(DOMAINS)*BASE_EPISODES_PER_DOMAIN} total")

base_results = []
for di, domain in enumerate(DOMAINS):
    for ep in range(BASE_EPISODES_PER_DOMAIN):
        seed = di * 100 + ep + 1  # same seeds as LoRA run
        print(f"  [{domain}] seed={seed}", end=" ... ")

        result = run_episode(base_m, base_t, seed)
        result["domain"] = domain
        result["seed"] = seed
        base_results.append(result)

        if "error" in result:
            print(f"ERROR: {result['error']}")
        else:
            print(f"DQ={result['final_dq']:.4f} R={result['avg_reward']:.4f}")

del base_m, base_t
torch.cuda.empty_cache()
print("Base model inference complete. GPU freed.")

In [ ]:
# Cell 9: Summarize and save results
import os
import sys
import json
import numpy as np
from datetime import datetime

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def summarize(results):
    """Aggregate episode results into summary statistics."""
    valid = [r for r in results if "error" not in r]
    if not valid:
        return {"error": "no valid episodes"}
    return {
        "final_dq_mean": float(np.mean([r["final_dq"] for r in valid])),
        "final_dq_std": float(np.std([r["final_dq"] for r in valid])),
        "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid])),
        "dq_improvement_mean": float(np.mean([r["dq_improvement"] for r in valid])),
        "steps_mean": float(np.mean([r["steps"] for r in valid])),
        "n_episodes": len(valid),
        "per_domain": {
            d: {
                "final_dq_mean": float(np.mean([r["final_dq"] for r in valid if r.get("domain") == d])),
                "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid if r.get("domain") == d])),
                "n": len([r for r in valid if r.get("domain") == d]),
            }
            for d in DOMAINS
            if any(r.get("domain") == d for r in valid)
        },
    }


output = {
    "task": "cleaning",
    "datasage_lora": summarize(lora_results),
    "base_model": summarize(base_results),
    "raw_lora": [
        {k: v for k, v in r.items() if k != "actions"}
        for r in lora_results
    ],
    "raw_base": [
        {k: v for k, v in r.items() if k != "actions"}
        for r in base_results
    ],
    "metadata": {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "lora_repo": LORA_REPO,
        "base_model": BASE_MODEL,
        "env_url": ENV_URL,
        "domains": DOMAINS,
        "episodes_per_domain": EPISODES_PER_DOMAIN,
        "base_episodes_per_domain": BASE_EPISODES_PER_DOMAIN,
        "max_steps": MAX_STEPS,
    },
}

OUTPUT_FILE = "cleaning_results.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved: {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE)/1024:.1f} KB)")

# Print quick comparison
ls = output["datasage_lora"]
bs = output["base_model"]
if "error" not in ls and "error" not in bs:
    print(f"\n  LoRA  DQ={ls['final_dq_mean']:.4f}  R={ls['avg_reward_mean']:.4f}  ({ls['n_episodes']} eps)")
    print(f"  Base  DQ={bs['final_dq_mean']:.4f}  R={bs['avg_reward_mean']:.4f}  ({bs['n_episodes']} eps)")
    print(f"  Delta DQ={ls['final_dq_mean']-bs['final_dq_mean']:+.4f}  R={ls['avg_reward_mean']-bs['avg_reward_mean']:+.4f}")

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print(f"\nDownloaded {OUTPUT_FILE} to your browser.")